## Inspect Missing Values in Ames Housing Dataset

In [9]:
from pathlib import Path
import pandas as pd

# Locate the Ames Housing dataset CSV file
csv_path = Path("data/train.csv")

if not csv_path.exists():
    csv_path = Path("../data/train.csv")

# Load the dataset
df = pd.read_csv(
    csv_path,
    keep_default_na=False,
    na_values=["NA", ""]
)

# Verify masonry veneer values
print("Missing MasVnrType:", df["MasVnrType"].isna().sum())
print("None category:", (df["MasVnrType"] == "None").sum())

print(
    "None with positive area:",
    ((df["MasVnrType"] == "None") & (df["MasVnrArea"] > 0)).sum()
)

print("Missing MasVnrArea:", df["MasVnrArea"].isna().sum())

# Count missing values in each column
missing_count = df.isna().sum()

# Calculate missing percentages
missing_percent = (missing_count / len(df)) * 100

# Create a missing-value report
missing_report = pd.DataFrame({
    "Missing" : missing_count,
    "Missing (%)": missing_percent
}).sort_values(by="Missing", ascending=False)

print("Dataset shape:", df.shape)

# Display the 15 columns with the most missing values
display(missing_report[missing_report["Missing"] > 0].head(15))

Missing MasVnrType: 8
None category: 864
None with positive area: 5
Missing MasVnrArea: 8
Dataset shape: (1460, 81)


,Missing,Missing (%)
PoolQC,1453,99.520548
MiscFeature,1406,96.301370
Alley,1369,93.767123
Fence,1179,80.753425
FireplaceQu,690,47.260274
LotFrontage,259,17.739726
GarageFinish,81,5.547945
GarageQual,81,5.547945
GarageYrBlt,81,5.547945
GarageType,81,5.547945


## Investigate if PoolQC is Missing When PoolArea is Zero

In [10]:
# Identify missing pool quality values
missing_pool = df["PoolQC"].isna()

# Identify houses with zero pool area
no_pool_area = df["PoolArea"] == 0

# Count houses satisfying both conditions
matched = (missing_pool & no_pool_area).sum()

print("Missing PoolQC:", missing_pool.sum())
print("Missing PoolQC and PoolArea == 0:", matched)

# Calculate the percentage of matching observations
percentage = (matched / missing_pool.sum()) * 100

print(f"Matching percentage: {percentage:.2f}%")

Missing PoolQC: 1453
Missing PoolQC and PoolArea == 0: 1453
Matching percentage: 100.00%


## Investigate GarageType Mismatches and Missing Values

In [11]:
# Identify missing garage type
missing_garage = df["GarageType"].isna()

# Identify houses with zero garage area
no_garage_area = df["GarageArea"] == 0

# Count matching conditions
matched_garage = (missing_garage & no_garage_area).sum()

# Find rows where the conditions disagree
mismatches = (missing_garage != no_garage_area).sum()

print("Missing GarageType:", missing_garage.sum())
print("Zero GarageArea:", no_garage_area.sum())
print("Both conditions:", matched_garage)
print("Mismatches:", mismatches)

Missing GarageType: 81
Zero GarageArea: 81
Both conditions: 81
Mismatches: 0


## Investigate MiscFeature Missing Values

In [12]:
# Check for missing miscellaneous features
missing_misc = df["MiscFeature"].isna()

# Check for zero miscellaneous feature value
zero_misc_value = df["MiscVal"] == 0

# Compare both conditions
comparison = pd.crosstab(
    missing_misc,
    zero_misc_value,
    rownames=["MiscFeature Missing"],
    colnames=["MiscVal is Zero"]
)

# Find houses with a miscellaneous feature but a zero value
unusual_misc = df.loc[
    df["MiscFeature"].notna() & (df["MiscVal"] == 0),
    ["Id", "MiscFeature", "MiscVal"]
]

display(comparison)
display(unusual_misc)

# Count inconsistencies
mismatches = (missing_misc != zero_misc_value).sum()

print("Mismatches:", mismatches)

MiscVal is Zero,False,True
MiscFeature Missing,,
False,52,2
True,0,1406


,Id,MiscFeature,MiscVal
873,874,Othr,0
1200,1201,Shed,0


Mismatches: 2


## Categorizing the Remaining Missing Values

In [13]:
# Identify columns containing missing values
missing_cols = df.columns[df.isna().any()]

# Select only columns with missing values
missing_df = df[missing_cols]

# Separate missing columns by data type
numeric_missing = missing_df.select_dtypes(
    include=['number']
).columns.tolist()

categorical_missing = missing_df.select_dtypes(
    exclude=['number']
).columns.tolist()

print("Numerical features with missing values")
print(numeric_missing)

print("\nCategorical features with missing values:")
print(categorical_missing)

print("\nTotal:", len(missing_cols))

Numerical features with missing values
['LotFrontage', 'MasVnrArea', 'GarageYrBlt']

Categorical features with missing values:
['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']

Total: 19


## Investigate MasVnrType and MasVnrArea Missing Values

In [14]:
# Identify missing masonry veneer type
missing_type = df["MasVnrType"].isna()

# Check related masonry veneer area
area_zero = df["MasVnrArea"] == 0
area_positive = df["MasVnrArea"] > 0
area_missing = df["MasVnrArea"].isna()

print("Missing MasVnrType:", missing_type.sum())

print("Missing type + Zero area:",
      (missing_type & area_zero).sum())

print("Missing type + Positive area:",
      (missing_type & area_positive).sum())

print("Missing type + Missing area:",
      (missing_type & area_missing).sum())

Missing MasVnrType: 8
Missing type + Zero area: 0
Missing type + Positive area: 0
Missing type + Missing area: 8


## Verify the Raw Masonry Values

In [15]:
# Reload the CSV without automatic missing value conversion
raw_df = pd.read_csv(csv_path, keep_default_na=False)

# Inspect original values that pandas interpreted as missing
print("Original masonry type values:")
print(
    raw_df.loc[missing_type, "MasVnrType"].value_counts(dropna=False)
)

# Inspect the 13 unusual observations
unusual_rows = missing_type &  (area_positive | area_missing)

display(
    raw_df.loc[
        unusual_rows,
        ["Id", "MasVnrType", "MasVnrArea"]
    ]
)

Original masonry type values:
MasVnrType
NA    8
Name: count, dtype: int64


,Id,MasVnrType,MasVnrArea
234,235,NA,NA
529,530,NA,NA
650,651,NA,NA
936,937,NA,NA
973,974,NA,NA
977,978,NA,NA
1243,1244,NA,NA
1278,1279,NA,NA


## Investigate Basement Features with Missing Values

In [16]:
# Basement-related categorical features
basement_cols = [
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2"
]

# Identify houses with no recorded basement area
no_basement = df["TotalBsmtSF"] == 0

print("Houses with zero basement area:", no_basement.sum())

# Investigate missing basement information
for col in basement_cols:
    missing = df[col].isna()

    print(f"\n{col}")
    print("Total missing:", missing.sum())
    print("Missing + No basement:", (missing & no_basement).sum())
    print("Missing + Has basement:", (missing & ~no_basement).sum())

Houses with zero basement area: 37

BsmtQual
Total missing: 37
Missing + No basement: 37
Missing + Has basement: 0

BsmtCond
Total missing: 37
Missing + No basement: 37
Missing + Has basement: 0

BsmtExposure
Total missing: 38
Missing + No basement: 37
Missing + Has basement: 1

BsmtFinType1
Total missing: 37
Missing + No basement: 37
Missing + Has basement: 0

BsmtFinType2
Total missing: 38
Missing + No basement: 37
Missing + Has basement: 1


## Inspect the Basement Exceptions

In [18]:
# Select houses with a basement but missing related information
basement_exceptions = df.loc[
    (df["TotalBsmtSF"] > 0) &
    (
        df["BsmtExposure"].isna() |
        df["BsmtFinType2"].isna()
    ),
    [
        "Id",
        "TotalBsmtSF",
        "BsmtExposure",
        "BsmtFinType1",
        "BsmtFinType2",
        "BsmtFinSF1",
        "BsmtFinSF2"
    ]
]

display(basement_exceptions)

,Id,TotalBsmtSF,BsmtExposure,BsmtFinType1,BsmtFinType2,BsmtFinSF1,BsmtFinSF2
332,333,3206,No,GLQ,NaN,1124,479
948,949,936,NaN,Unf,Unf,0,0


## Investigate the Actual Basement Exposure

In [19]:
# Count the different basement exposure categories
exposure_counts = df["BsmtExposure"].value_counts(
    dropna=False
)

display(exposure_counts)

BsmtExposure
No     953
Av     221
Gd     134
Mn     114
NaN     38
Name: count, dtype: int64